# RAG end-to-end: Base vs V2 + Qwen2.5-7B (Kaggle)

Notebook chạy **full pipeline** giống `scripts/run_ragas_retriever_comparison.py`, thay **Gemini API** bằng **Qwen2.5-7B-Instruct local**.

**Tương thích Save Version** (Run All, không cần Restart session).

## Kaggle setup

1. Bật **GPU T4** + **Internet**
2. Add dataset `training-data` + `models` (E2E) hoặc `ragas-final` (RAGAS-only)
3. **Save Version** / Run All — cell 1 tự cài dependency đúng version

Output: `/kaggle/working/ragas_eval/`

In [ ]:
# CELL 0 — Cài dependency (Save Version: không cần restart)
# Kaggle image thường có langchain-community 0.4+ → ragas lỗi vertexai import.
import subprocess
import sys


def pip_install(*packages: str, force: bool = False) -> None:
    cmd = [sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir"]
    if force:
        cmd.append("--force-reinstall")
    cmd.extend(packages)
    subprocess.check_call(cmd)


# 1) Gỡ bản 0.4+ nếu có, cài lại langchain stack 0.3.x
pip_install("langchain-community==0.3.27", "langchain==0.3.25", "langchain-core==0.3.65", force=True)

# 2) Cài ragas + phần còn lại
pip_install(
    "ragas==0.2.14",
    "langchain-huggingface>=0.1.2",
    "datasets>=2.19",
    "transformers>=4.44",
    "accelerate>=0.33",
    "bitsandbytes>=0.43",
    "sentence-transformers>=3.0",
    "einops",
)

import langchain_community

ver = langchain_community.__version__
assert ver.startswith("0.3."), f"langchain-community phải 0.3.x, đang là {ver}"

from ragas import evaluate  # noqa: F401 — smoke test import

print(f"OK deps | langchain-community={ver} | python={sys.version.split()[0]}")

## 1. Cấu hình

In [ ]:
import gc
import json
import random
import time
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# --- Giống scripts/run_ragas_retriever_comparison.py ---
RANDOM_SEED = 42
SOURCE = "training_data_pldc"
SAMPLE_SIZE = 30
TOP_K = 5
ENCODE_BATCH_SIZE = 32
SMOKE_SAMPLE_SIZE = None  # vd: 3 để thử nhanh

# False = full E2E | True = chỉ RAGAS trên ragas-final/*.jsonl
RAGAS_ONLY = False

QWEN_MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
QWEN_MAX_NEW_TOKENS = 1024  # script Gemini max_tokens=1024
RAGAS_EMBEDDING_ID = "bkai-foundation-models/vietnamese-bi-encoder"
BASE_ENCODER_ID = "bkai-foundation-models/vietnamese-bi-encoder"

RAGAS_BATCH_SIZE = 1
RAGAS_MAX_WORKERS = 1
RAGAS_TIMEOUT = 240
ANSWER_RELEVANCY_STRICTNESS = 3  # mặc định RAGAS, giống script

SYSTEM_PROMPT = """Bạn là trợ lý học tập cho giáo trình lý luận chính trị Việt Nam.

Chỉ trả lời dựa trên NGỮ CẢNH được cung cấp. Nếu ngữ cảnh không đủ thông tin,
hãy nói rõ rằng không tìm thấy thông tin trong ngữ cảnh. Trả lời ngắn gọn,
đúng trọng tâm, bằng tiếng Việt.

NGỮ CẢNH:
{context}"""

KAGGLE_USER = "dangvy1507"
KAGGLE_INPUT_BASE = Path(f"/kaggle/input/datasets/{KAGGLE_USER}")
KAGGLE_TRAINING_DIR = KAGGLE_INPUT_BASE / "training-data"
KAGGLE_V2_MODEL_DIR = KAGGLE_INPUT_BASE / "models" / "bi_encoder_hnm_v2" / "vietnamese-bi-encoder-v2-hnm"
KAGGLE_RAGAS_DIR = KAGGLE_INPUT_BASE / "ragas-final"

random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

IS_KAGGLE = Path("/kaggle").is_dir()
if IS_KAGGLE:
    REPO_ROOT = Path("/kaggle/working")
    TRAINING_BASE = KAGGLE_TRAINING_DIR
    V2_MODEL_PATH = str(KAGGLE_V2_MODEL_DIR)
    print("Environment: Kaggle")
else:
    REPO_ROOT = Path.cwd()
    if (REPO_ROOT / "Notebook").is_dir() and not (REPO_ROOT / "data").is_dir():
        REPO_ROOT = REPO_ROOT.parent
    TRAINING_BASE = REPO_ROOT / "data" / "training_data"
    V2_MODEL_PATH = str(
        REPO_ROOT / "bi-encoder-finetuned" / "models" / "bi_encoder_hnm_v2" / "vietnamese-bi-encoder-v2-hnm"
    )
    print("Environment: local")

OUT_DIR = REPO_ROOT / "ragas_eval"
OUT_DIR.mkdir(parents=True, exist_ok=True)

EMBED_MODELS = [
    ("base_vietnamese_bi_encoder", BASE_ENCODER_ID),
    ("v2_hard_negative", V2_MODEL_PATH),
]

effective_sample_size = SMOKE_SAMPLE_SIZE or SAMPLE_SIZE
device = "cuda" if torch.cuda.is_available() else "cpu"

print(f"Mode          : {'RAGAS_ONLY' if RAGAS_ONLY else 'END_TO_END'}")
print(f"Device        : {device}")
if device == "cuda":
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
print(f"Training base : {TRAINING_BASE}")
print(f"V2 model      : {V2_MODEL_PATH}")
print(f"Output        : {OUT_DIR}")

## 2. Helpers (copy logic từ script)

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import AnswerRelevancy, context_precision, context_recall, faithfulness
from ragas.run_config import RunConfig
from sentence_transformers import SentenceTransformer


def clean_text(text: str | None) -> str:
    return " ".join(str(text or "").split())


def load_jsonl(path: Path) -> list[dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def write_jsonl(path: Path, rows: list[dict]) -> None:
    with open(path, "w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def load_all_audits(training_base: Path) -> list[dict]:
    preferred_dirs = ["training_data_lsd", "training_data_pldc", "training_data_th"]
    files = [training_base / name / "qa_audit.jsonl" for name in preferred_dirs]
    files = [path for path in files if path.is_file()]
    if not files:
        files = sorted(training_base.glob("**/qa_audit.jsonl"))
    if not files:
        raise FileNotFoundError(f"Không tìm thấy qa_audit.jsonl dưới {training_base}")
    rows = []
    for path in files:
        source = path.parent.name
        for row in load_jsonl(path):
            row["_source"] = source
            rows.append(row)
    return rows


def chapter_key(row: dict) -> tuple[str, str]:
    return row.get("_source", ""), row.get("chapter", "Unknown")


def split_by_chapter_3way(
    rows: list[dict], val_ratio: float = 0.15, test_ratio: float = 0.15, seed: int = 42
) -> tuple[list[dict], list[dict], list[dict]]:
    by_key = defaultdict(list)
    for row in rows:
        by_key[chapter_key(row)].append(row)
    keys = list(by_key.keys())
    rng = random.Random(seed)
    rng.shuffle(keys)
    val_target = int(len(rows) * val_ratio)
    test_target = int(len(rows) * test_ratio)
    train_rows, val_rows, test_rows = [], [], []
    for key in keys:
        bucket = by_key[key]
        if len(val_rows) < val_target:
            val_rows.extend(bucket)
        elif len(test_rows) < test_target:
            test_rows.extend(bucket)
        else:
            train_rows.extend(bucket)
    return train_rows, val_rows, test_rows


def select_eval_rows(rows: list[dict], source: str, sample_size: int, seed: int) -> list[dict]:
    selected = [row for row in rows if row.get("_source") == source]
    if not selected:
        raise ValueError(f"Không có sample test cho source={source}")
    rng = random.Random(seed)
    rng.shuffle(selected)
    return selected[: min(sample_size, len(selected))]


def build_corpus(rows: list[dict], source: str) -> list[str]:
    texts, seen = [], set()
    for row in rows:
        if row.get("_source") != source:
            continue
        text = clean_text(row.get("positive"))
        if text and text not in seen:
            texts.append(text)
            seen.add(text)
    return texts


def retrieve_contexts(
    model_path: str, queries: list[str], corpus: list[str], top_k: int, batch_size: int
) -> tuple[list[list[str]], dict]:
    model = SentenceTransformer(model_path)
    started = time.perf_counter()
    query_emb = model.encode(queries, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
    corpus_emb = model.encode(corpus, batch_size=batch_size, normalize_embeddings=True, show_progress_bar=True)
    encode_seconds = time.perf_counter() - started
    query_emb = np.asarray(query_emb, dtype=np.float32)
    corpus_emb = np.asarray(corpus_emb, dtype=np.float32)
    scores = query_emb @ corpus_emb.T
    k = min(top_k, len(corpus))
    top_idx_unsorted = np.argpartition(-scores, kth=k - 1, axis=1)[:, :k]
    top_scores = np.take_along_axis(scores, top_idx_unsorted, axis=1)
    order = np.argsort(-top_scores, axis=1)
    top_idx = np.take_along_axis(top_idx_unsorted, order, axis=1)
    contexts = [[corpus[int(idx)] for idx in row] for row in top_idx]
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return contexts, {"encode_seconds": encode_seconds, "queries": len(queries), "corpus": len(corpus)}


def generate_answers_qwen(model, tokenizer, questions: list[str], contexts: list[list[str]]) -> list[str]:
    answers = []
    for idx, (question, ctxs) in enumerate(zip(questions, contexts), 1):
        context = "\n\n".join(f"[Đoạn {i}] {ctx}" for i, ctx in enumerate(ctxs, 1))
        prompt_text = SYSTEM_PROMPT.format(context=context) + f"\n\nCÂU HỎI:\n{question}"
        messages = [
            {"role": "system", "content": "Bạn là trợ lý học tập cho giáo trình lý luận chính trị Việt Nam."},
            {"role": "user", "content": prompt_text},
        ]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        print(f"Generating answer {idx}/{len(questions)}")
        with torch.no_grad():
            output_ids = model.generate(
                **inputs,
                max_new_tokens=QWEN_MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        generated = output_ids[0, inputs["input_ids"].shape[1] :]
        answers.append(clean_text(tokenizer.decode(generated, skip_special_tokens=True)))
    return answers


def run_ragas_eval(rows, answers, contexts, llm, embeddings) -> dict:
    records = [
        {
            "question": clean_text(row["query"]),
            "answer": answer,
            "contexts": ctxs,
            "ground_truth": clean_text(row["positive"]),
        }
        for row, answer, ctxs in zip(rows, answers, contexts)
    ]
    dataset = Dataset.from_list(records)
    result = evaluate(
        dataset,
        metrics=[
            faithfulness,
            AnswerRelevancy(strictness=ANSWER_RELEVANCY_STRICTNESS),
            context_precision,
            context_recall,
        ],
        llm=llm,
        embeddings=embeddings,
        run_config=RunConfig(
            timeout=RAGAS_TIMEOUT, max_workers=RAGAS_MAX_WORKERS, max_retries=2, max_wait=20
        ),
        raise_exceptions=False,
        batch_size=RAGAS_BATCH_SIZE,
    )
    scores = dict(getattr(result, "_repr_dict", {}))
    if not scores:
        scores = result.to_pandas().select_dtypes(include=["number"]).mean().to_dict()
    return scores


def run_ragas_from_jsonl(path: Path, llm, embeddings) -> tuple[dict, list[dict]]:
    records = load_jsonl(path)
    if SMOKE_SAMPLE_SIZE:
        records = records[:SMOKE_SAMPLE_SIZE]
    dataset = Dataset.from_list(
        [{"question": r["question"], "answer": r["answer"], "contexts": r["contexts"], "ground_truth": r["ground_truth"]} for r in records]
    )
    result = evaluate(
        dataset,
        metrics=[faithfulness, AnswerRelevancy(strictness=ANSWER_RELEVANCY_STRICTNESS), context_precision, context_recall],
        llm=llm,
        embeddings=embeddings,
        run_config=RunConfig(timeout=RAGAS_TIMEOUT, max_workers=RAGAS_MAX_WORKERS, max_retries=2, max_wait=20),
        raise_exceptions=False,
        batch_size=RAGAS_BATCH_SIZE,
    )
    scores = dict(getattr(result, "_repr_dict", {}))
    if not scores:
        scores = result.to_pandas().select_dtypes(include=["number"]).mean().to_dict()
    return scores, records

print("OK helpers")

## 3. Load Qwen + embedding (generate + RAGAS judge)

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEmbeddings, HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

print(f"Loading {QWEN_MODEL_ID} ...")
tokenizer = AutoTokenizer.from_pretrained(QWEN_MODEL_ID)
qwen_model = AutoModelForCausalLM.from_pretrained(
    QWEN_MODEL_ID, quantization_config=bnb_config, device_map="auto", torch_dtype=torch.float16
)
gen_pipeline = pipeline(
    "text-generation", model=qwen_model, tokenizer=tokenizer,
    max_new_tokens=QWEN_MAX_NEW_TOKENS, do_sample=False, return_full_text=False,
)
ragas_llm = ChatHuggingFace(llm=HuggingFacePipeline(pipeline=gen_pipeline))
ragas_embeddings = HuggingFaceEmbeddings(
    model_name=RAGAS_EMBEDDING_ID, model_kwargs={"device": device}, encode_kwargs={"normalize_embeddings": True}
)
print("OK Qwen + embeddings")

## 4. Chuẩn bị eval set (E2E) hoặc load artifact (RAGAS-only)

In [ ]:
split_info = {"mode": "ragas_only" if RAGAS_ONLY else "end_to_end", "llm": QWEN_MODEL_ID}

if not RAGAS_ONLY:
    all_rows = load_all_audits(TRAINING_BASE)
    train_rows, val_rows, test_rows = split_by_chapter_3way(all_rows, seed=RANDOM_SEED)
    eval_rows = select_eval_rows(test_rows, SOURCE, effective_sample_size, RANDOM_SEED)
    corpus = build_corpus(all_rows, SOURCE)
    questions = [clean_text(row["query"]) for row in eval_rows]
    split_info.update({
        "total_pairs": len(all_rows), "train": len(train_rows), "val": len(val_rows), "test": len(test_rows),
        "source": SOURCE, "eval_samples": len(eval_rows), "top_k": TOP_K, "corpus_size": len(corpus),
        "intent_distribution": dict(Counter(row.get("intent_type", "unknown") for row in eval_rows)),
    })
    print(json.dumps(split_info, ensure_ascii=False, indent=2))
else:
    eval_rows, corpus, questions = None, None, None
    print("RAGAS_ONLY: bỏ qua split/retrieval/generate")

## 5. Eval loop — 2 embedding models

In [ ]:
summaries = []

for name, model_path in EMBED_MODELS:
    print(f"\n{'=' * 60}\n=== {name}: {model_path} ===\n{'=' * 60}")

    if RAGAS_ONLY:
        jsonl_path = KAGGLE_RAGAS_DIR / f"{name}_generated_records.jsonl" if IS_KAGGLE else OUT_DIR / f"{name}_generated_records.jsonl"
        if not jsonl_path.is_file():
            jsonl_path = KAGGLE_RAGAS_DIR / f"{name}_generated_records.jsonl"
        if not jsonl_path.is_file() and not IS_KAGGLE:
            jsonl_path = REPO_ROOT / "artifacts" / "ragas_retriever_comparison_pldc_30_retry" / f"{name}_generated_records.jsonl"
        ragas_scores, records = run_ragas_from_jsonl(jsonl_path, ragas_llm, ragas_embeddings)
        write_jsonl(OUT_DIR / f"{name}_records.jsonl", records)
        summary = {"model": name, "model_path": model_path, "queries": len(records), **ragas_scores}
    else:
        contexts, retrieval_stats = retrieve_contexts(model_path, questions, corpus, TOP_K, ENCODE_BATCH_SIZE)
        answers = generate_answers_qwen(qwen_model, tokenizer, questions, contexts)
        pre_records = [
            {"question": clean_text(r["query"]), "answer": a, "contexts": c, "ground_truth": clean_text(r["positive"]), "model": name}
            for r, a, c in zip(eval_rows, answers, contexts)
        ]
        write_jsonl(OUT_DIR / f"{name}_generated_records.jsonl", pre_records)
        ragas_scores = run_ragas_eval(eval_rows, answers, contexts, ragas_llm, ragas_embeddings)
        summary = {"model": name, "model_path": model_path, **retrieval_stats, **ragas_scores}

    summaries.append(summary)
    print(json.dumps(summary, ensure_ascii=False, indent=2, default=str))

print("\nDone.")

## 6. Kết quả + delta V2 vs Base

In [ ]:
df = pd.DataFrame(summaries)
ragas_cols = ["faithfulness", "answer_relevancy", "context_precision", "context_recall"]
for col in ragas_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
if len(df) >= 2:
    base = df.iloc[0]
    for col in ragas_cols:
        if col in df.columns:
            df[f"delta_{col}_vs_base"] = df[col] - float(base[col])
display(df)

split_info["ragas_embedding"] = RAGAS_EMBEDDING_ID
with open(OUT_DIR / "ragas_summary.json", "w", encoding="utf-8") as f:
    json.dump({"split_info": split_info, "summaries": summaries}, f, ensure_ascii=False, indent=2, default=str)
df.to_csv(OUT_DIR / "ragas_summary.csv", index=False)
print(f"Saved: {OUT_DIR / 'ragas_summary.csv'}")